In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# EDIT THESE if your Drive path is different.
DRIVE_ROOT    = '/content/drive/MyDrive/midmamba'
DRIVE_RESULTS = f'{DRIVE_ROOT}/results'        # where to persist outputs across sessions
LOCAL_ROOT    = '/content/midmamba'             # working copy on /content (fast SSD)
LOCAL_RESULTS = f'{LOCAL_ROOT}/results'

# Make these visible to subsequent cells.
import os
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
os.environ['DRIVE_RESULTS'] = DRIVE_RESULTS
os.environ['LOCAL_ROOT'] = LOCAL_ROOT
os.environ['LOCAL_RESULTS'] = LOCAL_RESULTS

# 1) Copy code + configs + data from Drive to /content. Skip results/ so
#    we always start the run from a known-clean state on local disk;
#    we'll pull any prior results down explicitly in Cell 1b if you want resume.
!mkdir -p "$LOCAL_ROOT"
!rsync -a --info=stats2 \
    --exclude='results/' \
    --exclude='__pycache__/' \
    --exclude='.pytest_cache/' \
    --exclude='.git/' \
    --exclude='*.pyc' \
    "$DRIVE_ROOT/" "$LOCAL_ROOT/"

# 2) Make sure the persistent results directory exists on Drive.
!mkdir -p "$DRIVE_RESULTS"

%cd /content/midmamba
!ls
print(f"\nproject  → {LOCAL_ROOT}")
print(f"results  → {LOCAL_RESULTS}  (synced to {DRIVE_RESULTS})")

In [ ]:
!pip install --pre torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu128 -q

In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"

!pip install -U pip setuptools wheel ninja packaging
!pip install causal-conv1d --no-build-isolation
!pip install mamba-ssm --no-build-isolation

In [ ]:
!pip install \
    "apache-tvm-ffi<=0.1.9" \
    "quack-kernels>=0.3.4" \
    "tilelang==0.1.8" \
    torchao \
    databento \
    lightgbm \
    matplotlib \
    numpy \
    pandas \
    pyarrow \
    scikit-learn \
    seaborn \
    tqdm \
    wandb -q

In [ ]:
# Run after each phase. Atomic, additive — won't delete files on Drive.
import subprocess, os
def sync_to_drive():
    subprocess.check_call([
        'rsync', '-a', '--info=stats2',
        os.environ['LOCAL_RESULTS'] + '/',
        os.environ['DRIVE_RESULTS'] + '/',
    ])
    print(f"synced {os.environ['LOCAL_RESULTS']}/ → {os.environ['DRIVE_RESULTS']}/")

# Smoke-test it once with empty results.
os.makedirs(os.environ['LOCAL_RESULTS'], exist_ok=True)
sync_to_drive()

In [ ]:
import torch, subprocess
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('compute capability:', torch.cuda.get_device_capability(0))
    print('vram (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print(subprocess.check_output(['nvidia-smi']).decode())
!python -c "from mamba_ssm import Mamba2; import torch; m=Mamba2(d_model=64).cuda(); print('Mamba2 forward shape:', m(torch.randn(2,16,64).cuda()).shape)"

In [ ]:
!python -m pytest tests/ -q   # expect '76 passed'

In [ ]:
!python scripts/phase0_colab_checks.py

In [ ]:
!echo $LOCAL_RESULTS
!ls $LOCAL_RESULTS/phase1/stats/ 2>/dev/null || echo "phase1 not run yet or path differs"
!ls $LOCAL_RESULTS/ 2>/dev/null

In [ ]:
import json, os
qa_path = os.path.join(os.environ['LOCAL_RESULTS'], 'phase1', 'stats', 'qa_summary.json')
qa = json.load(open(qa_path))
print('\nExperiment cells:', list(qa['experiment_cells'].keys()))
print('Total rows after feature dropna:', qa['rows_total_after_feature_dropna'])

In [ ]:
import os
print("CWD:", os.getcwd())
print("LOCAL_RESULTS:", os.environ.get('LOCAL_RESULTS'))
print("data/ exists:", os.path.isdir('data'))
print("data/march2025:", os.path.isdir('data/march2025'))

In [ ]:
# Writes 3 horizons × 3 cells = 9 parquets to results/phase1/datasets/.
!python scripts/phase1_build_dataset.py --config configs/phase1.json

import json
qa = json.load(open('results/phase1/stats/qa_summary.json'))
print('\nExperiment cells:', list(qa['experiment_cells'].keys()))
print('Total rows after feature dropna:', qa['rows_total_after_feature_dropna'])
print('\nTrain/test row counts per (horizon, cell):')
for h, cells in qa['split_summary'].items():
    for c, v in cells.items():
        print(f"  {h:5s} {c}: train={v['train_rows']:>10,}  test={v['test_rows']:>10,}")

sync_to_drive()

In [ ]:
!python scripts/phase2_train_lightgbm.py --config configs/phase2.json

import json
m = json.load(open('results/phase2/lightgbm_metrics.json'))
print('\nLightGBM macro-F1 grid:')
print(f"{'horizon':<8}{'A':>10}{'B':>10}{'D':>10}")
for h, payload in m['horizons'].items():
    cells = payload['cells']
    row = [h] + [
        f"{cells.get(c, {}).get('test_macro_f1', 'NA'):>10.4f}"
        if isinstance(cells.get(c, {}).get('test_macro_f1'), (int, float))
        else f"{'NA':>10}"
        for c in ['A', 'B', 'D']
    ]
    print(''.join(row))

sync_to_drive()

In [ ]:
# Defaults from configs/phase3.json:
#   d_model=128, n_layers=3, seq_len=512, batch_size=64, epochs=5
#   d_state=256, d_conv=8, mlp_expand=2 (SwiGLU FFN per block)
#   pool_mode='gated_attention'
#   regression_head=True, reg_loss_weight=0.2 (multi-task with ret_h{H})
# Expect ~10–30 min per (horizon, cell) on the RTX Pro 6000.
!python scripts/phase3_train_mamba.py --config configs/phase3.json

import json
m = json.load(open('results/phase3/mamba_metrics.json'))
print('device:', m['device'])
print('mamba_kwargs:', m['config'].get('mamba_kwargs'))
print('reg_loss_weight:', m['config'].get('reg_loss_weight'))
for h, payload in m['runs'].items():
    print(f'\n--- {h} ---')
    for cell, v in payload['cells'].items():
        if 'error' in v:
            print(f'  {cell}: ERROR {v["error"]}'); continue
        tr = v.get('train_macro_f1')
        te = v.get('test_macro_f1')
        n_te = v.get('n_test_windows_evaluated', v.get('n_test_windows', 0))
        print(f"  {cell}: train_F1={tr:.4f}  test_F1={te if te is None else f'{te:.4f}'}  n_test_windows={n_te}")

sync_to_drive()

In [ ]:
# Identical PnL math, threshold sweep, hold period, spread cost for both models.
import json, pathlib, subprocess

base_cfg = json.loads(pathlib.Path('configs/phase4.json').read_text())
base_cfg['model_type'] = 'compare'

for h in [10, 50, 100]:
    for c in ['A', 'B', 'D']:
        cfg = dict(base_cfg)
        cfg.update({'horizon': h, 'cell': c, 'horizon_events': h})
        out = pathlib.Path(f'configs/_phase4_compare_h{h}_cell{c}.json')
        out.write_text(json.dumps(cfg, indent=2))
        subprocess.check_call(['python', 'scripts/phase4_backtest.py', '--config', str(out)])

sync_to_drive()

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob('results/phase4/compare_h*_cell*.json')):
    payload = json.load(open(f))
    h, c = payload['horizon'], payload['cell']
    table = {r['model']: r for r in payload.get('comparison_table', [])}
    lgbm = table.get('lightgbm', {})
    mamba = table.get('lobmambav2', {})
    info = payload.get('lobmambav2', {}).get('inference', {}) if isinstance(payload.get('lobmambav2'), dict) else {}
    rows.append({
        'h': h, 'cell': c,
        'lgbm_tau': lgbm.get('tau*'),
        'lgbm_trades': lgbm.get('n_trades'),
        'lgbm_bps': lgbm.get('mean_pnl_bps'),
        'lgbm_sharpe': lgbm.get('sharpe'),
        'mamba_tau': mamba.get('tau*'),
        'mamba_trades': mamba.get('n_trades'),
        'mamba_bps': mamba.get('mean_pnl_bps'),
        'mamba_sharpe': mamba.get('sharpe'),
        'mamba_warmup': info.get('warmup_rows'),
        'mamba_predicted': info.get('predicted_rows'),
    })
df = pd.DataFrame(rows).sort_values(['cell', 'h'])
print(df.to_string(index=False))

def winner(row):
    a, b = row['lgbm_bps'], row['mamba_bps']
    if a is None and b is None: return 'no-eligible'
    if a is None: return 'lobmambav2'
    if b is None: return 'lightgbm'
    return 'lobmambav2' if b > a else 'lightgbm'
df['better_bps'] = df.apply(winner, axis=1)
print('\nbps winner per (cell, horizon):')
print(df.pivot(index='cell', columns='h', values='better_bps'))

# Save the aggregated table to Drive too so it survives session disconnect.
df.to_csv(f"{os.environ['LOCAL_RESULTS']}/comparison_summary.csv", index=False)
sync_to_drive()

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

files = sorted(glob.glob('results/phase4/compare_h*_cell*.json'))
horizons = [10, 50, 100]
cells = ['A', 'B', 'D']
fig, axes = plt.subplots(len(horizons), len(cells), figsize=(14, 10), sharex=True)

for f in files:
    p = json.load(open(f))
    h, c = p['horizon'], p['cell']
    if h not in horizons or c not in cells: continue
    ax = axes[horizons.index(h), cells.index(c)]
    for tag, color in [('lightgbm', 'C0'), ('lobmambav2', 'C1')]:
        block = p.get(tag, {})
        if 'curves' not in block: continue
        taus = [c2['tau'] for c2 in block['curves']]
        bps = [c2['mean_pnl_bps'] for c2 in block['curves']]
        ax.plot(taus, bps, label=tag, color=color, marker='.')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(f'h={h}  cell={c}')
    ax.set_xlabel('threshold τ'); ax.set_ylabel('mean pnl (bps)')
    ax.legend(fontsize=8)
plt.tight_layout()
out_png = f"{os.environ['LOCAL_RESULTS']}/phase4/threshold_sweep.png"
plt.savefig(out_png, dpi=120)
plt.show()
print('saved →', out_png)
sync_to_drive()

In [ ]:
# Re-run Phase 3 with reg_loss_weight=0 to attribute the F1 delta you see vs the
# multi-task run. Writes to a separate output dir so checkpoints don't clash.
import json, pathlib
cfg = json.loads(pathlib.Path('configs/phase3.json').read_text())
cfg['reg_loss_weight'] = 0.0
cfg['regression_head'] = False
cfg['output_dir'] = 'results/phase3_no_reg'
pathlib.Path('configs/phase3_no_reg.json').write_text(json.dumps(cfg, indent=2))
!python scripts/phase3_train_mamba.py --config configs/phase3_no_reg.json

import json
ref = json.load(open('results/phase3/mamba_metrics.json'))
abl = json.load(open('results/phase3_no_reg/mamba_metrics.json'))
print(f"\n{'horizon':<8}{'cell':<6}{'with_reg':>12}{'no_reg':>12}{'delta':>10}")
for h, p in ref['runs'].items():
    for c, v in p['cells'].items():
        ref_f1 = v.get('test_macro_f1')
        abl_f1 = abl['runs'][h]['cells'].get(c, {}).get('test_macro_f1')
        if ref_f1 is not None and abl_f1 is not None:
            print(f"{h:<8}{c:<6}{ref_f1:>12.4f}{abl_f1:>12.4f}{ref_f1 - abl_f1:>+10.4f}")

sync_to_drive()